# One-packet flow context follow-up

This notebook describes Broad-remaining one-packet observations using the provenance-validated context artifacts. It does not read packet captures or recreate the pipeline. Interpret after real-data execution.

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from mawi_global_analysis.io import load_one_packet_context


def resolve_analysis_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'src' / 'mawi_global_analysis').is_dir():
            return candidate
    return Path.cwd()


root = Path(os.environ.get('MAWI_ANALYSIS_ROOT', resolve_analysis_root())).resolve()
dataset_id = os.environ.get('MAWI_DATASET_ID')
context_run_name = os.environ.get('MAWI_CONTEXT_RUN_NAME')
TOP_N = 15

if not dataset_id or not context_run_name:
    raise RuntimeError('MAWI_DATASET_ID と MAWI_CONTEXT_RUN_NAME を明示指定してください')

context_data = load_one_packet_context(dataset_id, context_run_name, root=root)
cohort = context_data.cohort.copy()
context = context_data.context.copy()
source_context = context_data.source_context.copy()
manifest = context_data.manifest


def display_table(frame):
    display(frame)


## 1. Provenance and cohort sanity checks

The manifest is the source of truth. This compact view intentionally omits the complete manifest.

In [ ]:
source_window = manifest['source_capture_window']
full_capture = manifest['full_capture']
source_run = manifest['source_run']
provenance_summary = pd.DataFrame([{
    'dataset': manifest['dataset_id'],
    'context run name': manifest['context_run_name'],
    'source run identity': f"{source_run['dataset_id']}/{source_run['run_name']}",
    'target capture window': f"{source_window['window_start']} to {source_window['window_end']}",
    'full DITL input identity/checksum': f"{full_capture['path']} ({full_capture['sha256']})",
    'cohort rows': len(cohort),
    'context rows': len(context),
    'source-context rows': len(source_context),
}])
display_table(provenance_summary)

linkage_columns = ['source_flow_id', 'target_timestamp']
for name, frame in [('cohort', cohort), ('context', context), ('source context', source_context)]:
    if frame.duplicated(linkage_columns).any():
        raise ValueError(f'{name} has duplicate source-flow linkage')
cohort_linkage = set(map(tuple, cohort[linkage_columns].to_numpy()))
if set(map(tuple, context[linkage_columns].to_numpy())) != cohort_linkage:
    raise ValueError('context linkage does not match the cohort')
if set(map(tuple, source_context[linkage_columns].to_numpy())) != cohort_linkage:
    raise ValueError('source-context linkage does not match the cohort')
if cohort['source_flow_id'].nunique() != len(cohort):
    raise ValueError('cohort source_flow_id values are not one-to-one')

analysis = cohort.merge(context, on=linkage_columns, how='inner', validate='one_to_one').merge(
    source_context, on=linkage_columns, how='inner', validate='one_to_one'
)
if len(analysis) != len(cohort):
    raise ValueError('artifact join unexpectedly lost cohort rows')
display_table(pd.DataFrame([{
    'cohort rows': len(cohort),
    'context rows': len(context),
    'source-context rows': len(source_context),
    'unique source_flow_id': cohort['source_flow_id'].nunique(),
    'one-to-one linkage verified': True,
}]))


## 2. Protocol composition

In [ ]:
protocol_names = {6: 'TCP', 17: 'UDP'}
protocol_summary = (analysis.assign(protocol_name=analysis['protocol'].map(protocol_names))
    .groupby('protocol_name', dropna=False).size().rename('count').reset_index())
if protocol_summary['protocol_name'].isna().any():
    raise ValueError('cohort contains a protocol outside the TCP/UDP flow scope')
protocol_summary['proportion'] = protocol_summary['count'] / len(analysis) if len(analysis) else 0.0
display_table(protocol_summary)


## 3. Exact TCP flag composition

Unknown or less common combinations remain separate observations.

In [ ]:
tcp_flags = (analysis.loc[analysis['protocol'].eq(6)]
    .groupby('tcp_flags_text', dropna=False).size().rename('count').reset_index()
    .sort_values('count', ascending=False))
tcp_flags['proportion'] = tcp_flags['count'] / len(analysis.loc[analysis['protocol'].eq(6)]) if (analysis['protocol'].eq(6)).any() else 0.0
display_table(tcp_flags)
if not tcp_flags.empty:
    tcp_flags.plot.bar(x='tcp_flags_text', y='count', legend=False, figsize=(7, 3.5), title='Exact TCP flag observations')
    plt.ylabel('cohort rows')
    plt.tight_layout()


## 4. Destination-port composition

`packet_dst_port` is the destination port of the matched target packet; it does not reinterpret canonical flow roles.

In [ ]:
for protocol, label in [(6, 'TCP'), (17, 'UDP')]:
    ports = (analysis.loc[analysis['protocol'].eq(protocol)]
        .groupby('packet_dst_port').size().rename('count').reset_index()
        .sort_values(['count', 'packet_dst_port'], ascending=[False, True]).head(TOP_N))
    print(f'{label}: top {TOP_N} observed target destination ports')
    display_table(ports)


## 5. One-packet size characteristics

Captured frame length is the length present in the capture record; original frame length is the wire-record length. Neither replaces canonical flow `frame_byte_count`.

In [ ]:
size_columns = ['captured_frame_length', 'original_frame_length', 'ip_total_length', 'transport_payload_length']
display_table(analysis[size_columns].describe().T)
analysis['captured_frame_length'].plot.hist(bins=30, figsize=(7, 3.5), title='Captured frame length')
plt.xlabel('bytes present in capture record')
plt.tight_layout()


## 6. Same-5-tuple 24h context

A value of `same_5tuple_packet_count_24h == 1` means no other matching bidirectional 5-tuple packet was observed in the supplied full capture. It does not establish absence of communication elsewhere.

In [ ]:
tuple_context_summary = pd.DataFrame([{
    'no additional same-5-tuple packet': int(analysis['same_5tuple_packet_count_24h'].eq(1).sum()),
    'additional same-direction packet(s)': int((analysis['forward_packet_count_24h'] > 1).sum()),
    'reverse-direction packet(s)': int((analysis['reverse_packet_count_24h'] > 0).sum()),
}])
display_table(tuple_context_summary)


## 7. Temporal context

These are observed timing relationships, not TCP-connection assertions.

In [ ]:
temporal_columns = ['previous_same_5tuple_gap_seconds', 'next_same_5tuple_gap_seconds', 'nearest_reverse_before_gap_seconds', 'nearest_reverse_after_gap_seconds']
display_table(analysis[temporal_columns].describe().T)
nearby_rows = []
for seconds in (1, 10, 60, 300):
    nearby_rows.append({
        'seconds': seconds,
        'same-tuple before count': int((analysis[f'same_5tuple_before_{seconds}s'] > 0).sum()),
        'same-tuple after count': int((analysis[f'same_5tuple_after_{seconds}s'] > 0).sum()),
    })
display_table(pd.DataFrame(nearby_rows))


## 8. 15-minute window-boundary context

A candidate has the same 5-tuple observed outside the 15-minute window. This is only a window-boundary context candidate.

In [ ]:
window_start = pd.Timestamp(source_window['window_start'])
window_end = pd.Timestamp(source_window['window_end'])
for column in ['previous_same_5tuple_timestamp', 'next_same_5tuple_timestamp']:
    analysis[f'{column}_utc'] = pd.to_datetime(analysis[column], unit='s', utc=True)
additional_inside_window = ((analysis['previous_same_5tuple_timestamp_utc'].ge(window_start) & analysis['previous_same_5tuple_timestamp_utc'].lt(window_end)) |
    (analysis['next_same_5tuple_timestamp_utc'].ge(window_start) & analysis['next_same_5tuple_timestamp_utc'].lt(window_end)))
if additional_inside_window.any():
    raise ValueError('an additional same-tuple packet unexpectedly falls inside the source window')
outside_window = ((analysis['previous_same_5tuple_timestamp_utc'].notna() & (analysis['previous_same_5tuple_timestamp_utc'].lt(window_start) | analysis['previous_same_5tuple_timestamp_utc'].ge(window_end))) |
    (analysis['next_same_5tuple_timestamp_utc'].notna() & (analysis['next_same_5tuple_timestamp_utc'].lt(window_start) | analysis['next_same_5tuple_timestamp_utc'].ge(window_end))))
display_table(pd.DataFrame([{
    'same 5-tuple observed outside the 15-minute window': int(outside_window.sum()),
    'no outside-window tuple observation': int((~outside_window).sum()),
}]))


## 9. Plain-SYN source context

Only rows marked applicable are summarized. No new threshold or behavioral label is introduced.

In [ ]:
applicable_source_context = analysis.loc[analysis['source_context_applicable'].eq(True)].copy()
source_context_columns = ['plain_syn_packet_count_5m', 'unique_target_count_5m', 'unique_dst_ip_count_5m', 'unique_dst_port_count_5m', 'plain_syn_packet_count_15m', 'unique_target_count_15m', 'unique_dst_ip_count_15m', 'unique_dst_port_count_15m', 'plain_syn_packet_count_1h', 'unique_target_count_1h', 'unique_dst_ip_count_1h', 'unique_dst_port_count_1h', 'plain_syn_packet_count_24h', 'unique_target_count_24h', 'unique_dst_ip_count_24h', 'unique_dst_port_count_24h']
display_table(pd.DataFrame([{
    'source-context applicable': len(applicable_source_context),
    'source-context not applicable': len(analysis) - len(applicable_source_context),
}]))
display_table(applicable_source_context[source_context_columns].describe().T)


## 10. Compact observational summary

Interpret after real-data execution.

In [ ]:
observational_summary = pd.DataFrame([{
    'cohort rows': len(analysis),
    'TCP rows': int(analysis['protocol'].eq(6).sum()),
    'UDP rows': int(analysis['protocol'].eq(17).sum()),
    'same-tuple context present': int(analysis['same_5tuple_packet_count_24h'].gt(1).sum()),
    'reverse-direction context present': int(analysis['reverse_packet_count_24h'].gt(0).sum()),
    'source context applicable': int(analysis['source_context_applicable'].eq(True).sum()),
    'same-tuple outside-window context present': int(outside_window.sum()),
}])
display_table(observational_summary)
